# WHT Time Series Forecasting - Exploration

This notebook explores the Walsh-Hadamard Transform (WHT) for time series forecasting.

In [ ]:
import sys
from pathlib import Path

# Add src to path (works from notebooks/ or project root)
for p in [Path.cwd(), Path.cwd().parent]:
    src = p / "src"
    if (src / "wht_forecast").exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
        break

import numpy as np
from wht_forecast import (
    build_normalized_hadamard,
    forecast_next_block,
    generate_synthetic_series,
    compute_metrics,
    naive_forecast,
    moving_average_forecast,
    linear_extrapolation_forecast,
)

## Generate synthetic data

In [ ]:
series = generate_synthetic_series(n=512, seed=42, noise_level=0.2)
print(f"Series length: {len(series)}")
print(f"Min: {series.min():.3f}, Max: {series.max():.3f}")

## Run WHT forecast

In [ ]:
block_size = 32
forecast, info = forecast_next_block(
    series, block_size=block_size, top_k=8, smooth_window=3
)

n_blocks = len(series) // block_size
actual_next = series[(n_blocks - 1) * block_size : n_blocks * block_size]
train_series = series[: (n_blocks - 1) * block_size]

metrics = compute_metrics(actual_next, forecast)
print(f"WHT Forecast - MAE: {metrics['MAE']:.4f}, RMSE: {metrics['RMSE']:.4f}")

## Compare with baselines

In [ ]:
fc_naive = naive_forecast(train_series, block_size)
fc_ma = moving_average_forecast(train_series, block_size, window=3)
fc_linear = linear_extrapolation_forecast(train_series, block_size)

for name, fc in [("Naive", fc_naive), ("Moving avg", fc_ma), ("Linear", fc_linear)]:
    m = compute_metrics(actual_next, fc)
    print(f"{name}: MAE={m['MAE']:.4f}, RMSE={m['RMSE']:.4f}")

## Visualize

In [ ]:
from wht_forecast.visualization import plot_results

A = build_normalized_hadamard(block_size)
forecast, info = forecast_next_block(train_series, A=A, block_size=block_size, top_k=8)

plot_results(train_series, forecast, actual_next, block_size, info)